In [24]:
"""
Credit Card Fraud Detection Analysis (BFIS)
Business Questions: Whiuch transaction patterns consistently signal fraud,
and what triggers should a real-time alert systum use?

Run:  python fraud_analysis.py  (put creditcard.csv in the same folder)
Outputs: charts/*.png and powerbi/*.csv
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("charts", exist_ok=True)
os.makedirs("powerbi", exist_ok=True)


In [25]:
# ---------------------------------------------------------------
# 1. LOAD + AUDIT
# ---------------------------------------------------------------

df = pd.read_csv("creditcard.csv")
print("Shape:", df.shape)
print("Nulls:", df.isna().sum().sum(), "| Duplicates:", df.duplicated().sum())
print("Fraud rate: %.3f%%" % (df["Class"].mean() * 100))

# Duplicates: keep them for the main analysis (identical amounts at the same
# second can be legitimate), but note the decision in the report.
# df = df.drop_duplicates()   # <- optional sensitivity check

Shape: (284807, 31)
Nulls: 0 | Duplicates: 1081
Fraud rate: 0.173%


In [26]:
# ---------------------------------------------------------------
# 2. FEATURE ENGINEERING
# ---------------------------------------------------------------
# Time = seconds since the FIRST transaction (not midnight) -> approximate hour

df["day"] = (df["Time"] // 86400).astype(int) + 1
df["hour"] = ((df["Time"] // 3600) % 24).astype(int)
df["time_bucket"] = pd.cut(
    df["hour"], bins=[-1, 5, 11, 17, 23],
    labels=["Night (0-5)", "Morning (6-11)", "Afternoon (12-17)", "Evening (18-23)"],
)
bins = [-0.01, 1, 10, 50, 100, 250, 500, 1000, 30000]
labels = ["<=1", "1-10", "10-50", "50-100", "100-250", "250-500", "500-1000", "1000+"]
df["amount_band"] = pd.cut(df["Amount"], bins=bins, labels=labels)
df["label"] = df["Class"].map({0: "Legit", 1: "Fraud"})

In [27]:
# ---------------------------------------------------------------
# 3. TIME-BASED PATTERNS
# ---------------------------------------------------------------

hourly = df.groupby("hour")["Class"].agg(txns="count", frauds="sum")
hourly["fraud_rate_pct"] = hourly["frauds"] / hourly["txns"] * 100
hourly.to_csv("powerbi/hourly_summary.csv")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(hourly.index, hourly["fraud_rate_pct"], color="#6C3BAA")
ax.axhline(df["Class"].mean() * 100, color="red", ls="--", label="Overall rate (0.17%)")
ax.set_xlabel("Hour of day (approx.)"); ax.set_ylabel("Fraud rate (%)")
ax.set_title("Fraud rate by hour: night hours spike"); ax.legend()
plt.tight_layout(); plt.savefig("charts/01_fraud_rate_by_hour.png", dpi=150); plt.close()

bucket = df.groupby("time_bucket", observed=True)["Class"].agg(txns="count", frauds="sum")
bucket["fraud_rate_pct"] = bucket["frauds"] / bucket["txns"] * 100
print("\nFraud by time bucket:\n", bucket.round(3))


Fraud by time bucket:
                     txns  frauds  fraud_rate_pct
time_bucket                                     
Night (0-5)        23934     124           0.518
Morning (6-11)     70912     118           0.166
Afternoon (12-17)  96435     134           0.139
Evening (18-23)    93526     116           0.124


In [28]:
# ---------------------------------------------------------------
# 4. AMOUNT DISTRIBUTIONS
# ---------------------------------------------------------------

print("\nAmount summary by class:\n", df.groupby("label")["Amount"].describe().round(2))

band = df.groupby("amount_band", observed=True)["Class"].agg(txns="count", frauds="sum")
band["fraud_rate_pct"] = band["frauds"] / band["txns"] * 100
band["share_of_all_fraud_pct"] = band["frauds"] / band["frauds"].sum() * 100
band.to_csv("powerbi/amount_band_summary.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for lab, c in [("Legit", "#4C78A8"), ("Fraud", "#E45756")]:
    axes[0].hist(np.log10(df.loc[df.label == lab, "Amount"] + 1), bins=50,
                 alpha=0.6, density=True, label=lab, color=c)
axes[0].set_xlabel("log10(Amount + 1)"); axes[0].set_ylabel("Density")
axes[0].set_title("Amount distribution (log scale)"); axes[0].legend()
axes[1].bar(band.index.astype(str), band["fraud_rate_pct"], color="#6C3BAA")
axes[1].set_title("Fraud rate by amount band"); axes[1].set_ylabel("Fraud rate (%)")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.savefig("charts/02_amount_patterns.png", dpi=150); plt.close()


Amount summary by class:
           count    mean     std  min   25%    50%     75%       max
label                                                              
Fraud     492.0  122.21  256.68  0.0  1.00   9.25  105.89   2125.87
Legit  284315.0   88.29  250.11  0.0  5.65  22.00   77.05  25691.16


In [29]:
# ---------------------------------------------------------------
# 5. "DEMOGRAPHIC" SUBSTITUTE: ANOMALOUS BEHAVIOUR PROFILE
#    (this dataset has no age/gender/city -> use V1..V28 deviations)
# ---------------------------------------------------------------

vcols = [f"V{i}" for i in range(1, 29)]
corr = df[vcols + ["Class"]].corr()["Class"].drop("Class").sort_values()
top = corr.reindex(corr.abs().sort_values(ascending=False).head(10).index)
print("\nTop features correlated with fraud:\n", top.round(3))

fig, ax = plt.subplots(figsize=(8, 4.5))
top.sort_values().plot.barh(ax=ax, color="#6C3BAA")
ax.set_title("Top 10 anonymized features by correlation with fraud")
plt.tight_layout(); plt.savefig("charts/03_feature_correlation.png", dpi=150); plt.close()

# Fraud vs legit mean for the top features (z-score style comparison)
prof = df.groupby("label")[list(top.index)].mean().T
prof["gap"] = prof["Fraud"] - prof["Legit"]
print("\nFeature means (Fraud vs Legit):\n", prof.round(2))


Top features correlated with fraud:
 V17   -0.326
V14   -0.303
V12   -0.261
V10   -0.217
V16   -0.197
V3    -0.193
V7    -0.187
V11    0.155
V4     0.133
V18   -0.111
Name: Class, dtype: float64

Feature means (Fraud vs Legit):
 label  Fraud  Legit   gap
V17    -6.67   0.01 -6.68
V14    -6.97   0.01 -6.98
V12    -6.26   0.01 -6.27
V10    -5.68   0.01 -5.69
V16    -4.14   0.01 -4.15
V3     -7.03   0.01 -7.05
V7     -5.57   0.01 -5.58
V11     3.80  -0.01  3.81
V4      4.54  -0.01  4.55
V18    -2.25   0.00 -2.25


In [30]:
# ---------------------------------------------------------------
# 6. ALERT RULES + EVALUATION (precision / recall)
# ---------------------------------------------------------------

rules = {
    "R1: Night hours (1-5 AM)":      df["hour"].between(1, 5),
    "R2: Micro amount (<= $1)":      df["Amount"] <= 1,
    "R3: High amount ($250-$1000)":  df["Amount"].between(250, 1000),
    "R4: V14 < -5":                  df["V14"] < -5,
    "R5: V17 < -5":                  df["V17"] < -5,
    "R6: V14 < -5 OR V17 < -5 OR V12 < -5":
        (df["V14"] < -5) | (df["V17"] < -5) | (df["V12"] < -5),
    "R7: Night AND V14 < -3":        df["hour"].between(1, 5) & (df["V14"] < -3),
}
rows = []
for name, flag in rules.items():
    tp = int((flag & (df["Class"] == 1)).sum())
    fp = int((flag & (df["Class"] == 0)).sum())
    fn = int(((~flag) & (df["Class"] == 1)).sum())
    rows.append({
        "rule": name, "alerts": int(flag.sum()), "frauds_caught": tp, "false_alarms": fp,
        "precision_pct": tp / max(tp + fp, 1) * 100,
        "recall_pct": tp / (tp + fn) * 100,
    })
perf = pd.DataFrame(rows)
perf.to_csv("powerbi/rule_performance.csv", index=False)
print("\nRule performance:\n", perf.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(perf["recall_pct"], perf["precision_pct"], s=90, color="#6C3BAA")
for _, r in perf.iterrows():
    ax.annotate(r["rule"].split(":")[0], (r["recall_pct"], r["precision_pct"]),
                xytext=(5, 5), textcoords="offset points")
ax.set_xlabel("Recall (% of fraud caught)"); ax.set_ylabel("Precision (% of alerts that are fraud)")
ax.set_title("Alert rules: precision vs recall trade-off")
plt.tight_layout(); plt.savefig("charts/04_rule_tradeoff.png", dpi=150); plt.close()


Rule performance:
                                 rule  alerts  frauds_caught  false_alarms  precision_pct  recall_pct
            R1: Night hours (1-5 AM)   16239            118         16121           0.73       23.98
            R2: Micro amount (<= $1)   30492            181         30311           0.59       36.79
        R3: High amount ($250-$1000)   19963             65         19898           0.33       13.21
                        R4: V14 < -5     541            324           217          59.89       65.85
                        R5: V17 < -5     328            265            63          80.79       53.86
R6: V14 < -5 OR V17 < -5 OR V12 < -5     623            379           244          60.83       77.03
              R7: Night AND V14 < -3     150            105            45          70.00       21.34


In [31]:
# ---------------------------------------------------------------
# 7. EXPORT FOR POWER BI
# ---------------------------------------------------------------

keep = ["Time", "day", "hour", "time_bucket", "Amount", "amount_band", "label", "Class",
        "V10", "V12", "V14", "V17"]
df[keep].to_csv("powerbi/fraud_clean.csv", index=False)
print("\nDone. See /charts and /powerbi")


Done. See /charts and /powerbi
